# Exploratory data analysis

Run after the first pipeline run: it reads `data/` and `reports/`. Open it in Jupyter or VS Code on your
machine with the project's requirements installed (`pip install -r requirements.txt jupyter`).

Questions: Is the sample balanced across rating bands? What gets rejected, and why? Which features carry
signal about rating? Are there missing values or outliers the model must handle?

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "params.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))      # the project's chessfeat package
reports = {name: json.loads((ROOT / "reports" / f"{name}.json").read_text())
           for name in ("ingest", "validation", "featurize")}
df = pd.read_parquet(ROOT / "data" / "features" / "features.parquet")
print(reports["ingest"]["month"], "-", len(df), "player rows from", df["game_id"].nunique(), "games")

## 1. Sampling and data quality

In [ ]:
print("Kept per rating band (by average rating of the two players):")
print(pd.Series(reports["ingest"]["kept_per_band"]))
print("\nRejected at ingest (header rules), by reason:")
print(pd.Series(reports["ingest"]["rejected"], dtype="int64").sort_values(ascending=False))
print("\nRejected at validation (full parse), by reason:")
print(pd.Series(reports["validation"]["counts"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df["rating"].plot.hist(bins=40, ax=ax)
ax.set(title="Player ratings in the sample", xlabel="Lichess blitz rating", ylabel="Players")
plt.show()

## 2. Missing values and ranges

In [ ]:
from chessfeat.features import FEATURES

summary = pd.DataFrame({"missing_share": df[FEATURES].isna().mean().round(3),
                        "min": df[FEATURES].min(), "median": df[FEATURES].median(), "max": df[FEATURES].max()})
summary

Expected: `acpl_endgame` is missing whenever the game never reached an endgame, and LightGBM handles
missing values natively, so they are kept as NaN rather than filled.

## 3. Which features relate to rating?

In [ ]:
corr = df[FEATURES + ["rating"]].corr(method="spearman")["rating"].drop("rating").sort_values()
fig, ax = plt.subplots(figsize=(6, 6))
corr.plot.barh(ax=ax)
ax.set(title="Spearman correlation with rating", xlabel="correlation")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.plot.scatter(x="acpl", y="rating", s=3, alpha=0.3, ax=axes[0], title="Average centipawn loss vs rating")
df.plot.scatter(x="best_move_rate", y="rating", s=3, alpha=0.3, ax=axes[1], title="Engine-best moves vs rating")
plt.show()

In [ ]:
from chessfeat.rules import band_of
import yaml

edges = yaml.safe_load((ROOT / "params.yaml").read_text())["bands"]
df["band"] = [band_of(r, edges) for r in df["rating"]]
df.groupby("band")[["acpl", "blunder_rate", "mean_move_time", "time_used_frac", "low_clock_moves"]].median().round(2)

## 4. Outliers

In [ ]:
print("Players with ACPL above 300:", int((df["acpl"] > 300).sum()))
print("Players who used under 10% of their clock:", int((df["time_used_frac"] < 0.1).sum()))
print("Players who lost on time:", int(df["lost_on_time"].sum()))
df.sort_values("acpl", ascending=False).head(5)[["game_id", "rating", "acpl", "plies", "result"]]

## 5. Findings

Write your findings here after running on the real month (do not copy numbers from any other run):

- Balance across bands:
- Main rejection reasons:
- Strongest features:
- Missing values and outliers, and how the pipeline handles them: